# Math 03: Calculus — The Chain Rule**Learning Objectives:**1. Differentiate composite functions using the Chain Rule2. Decompose nested functions into layers for differentiation3. Implement chain-rule differentiation computationally**Prerequisites:** Math 02 (Derivatives & Power Rule)**Estimated time:** 90 minutes**Textbook:** *Justin Skycak — Calculus* §1.5

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "math_03_calculus_chain_rule",    "level": 1,    "title": "Math 03: Calculus \u2014 The Chain Rule",    "prerequisites": [        "Math 02 (Derivatives & Power Rule)"    ],    "skills_taught": [        "lesson.math_03_calculus_chain_rule"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "math_04_calculus_local_extrema.ipynb"}SKILLS = [    {        "id": "lesson.math_03_calculus_chain_rule",        "title": "Lesson Math 03 Calculus Chain Rule",        "notebook": "math_03_calculus_chain_rule.ipynb",        "level": 1    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "math_03_calculus_chain_rule.ipynb",        "level": 1    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Math 02 (Derivatives & Power Rule).

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.math_03_calculus_chain_rule', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setup!pip install -q sympy matplotlib plotly ipywidgetsimport sympy as sp; sp.init_printing()import numpy as np, matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### The Chain Rule> 📖 *From Calculus §1.5:* The chain rule tells us how to differentiate compositions of functions.> Informally: 'forget' about the inside, differentiate the outside, then multiply by the derivative of the inside.For $y = f(g(x))$, let $u = g(x)$. Then:$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} = f'(g(x)) \cdot g'(x)$$**Intuition:** Derivatives cancel like fractions: $\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$### Examples| $y$ | $u$ (inside) | $dy/du$ | $du/dx$ | $dy/dx$ ||---|---|---|---|---|| $(3x+1)^5$ | $u=3x+1$ | $5u^4$ | $3$ | $15(3x+1)^4$ || $\sqrt{x^2+1}$ | $u=x^2+1$ | $\frac{1}{2\sqrt{u}}$ | $2x$ | $\frac{x}{\sqrt{x^2+1}}$ || $\sin(x^3)$ | $u=x^3$ | $\cos u$ | $3x^2$ | $3x^2\cos(x^3)$ |

In [ ]:
# Symbolic verificationimport sympy as spx = sp.Symbol('x')expressions = [(3*x+1)**5, sp.sqrt(x**2+1), sp.sin(x**3)]for expr in expressions:    print(f'd/dx [{expr}] = {sp.diff(expr, x)}')

### Multi-layer chainsFor deeply nested functions, apply the chain rule repeatedly:$$\frac{d}{dx}[f(g(h(x)))] = f'(g(h(x))) \cdot g'(h(x)) \cdot h'(x)$$**Example:** $y = \sin^2(3x) = [\sin(3x)]^2$- Layer 1: $u_1 = 3x \Rightarrow du_1/dx = 3$- Layer 2: $u_2 = \sin(u_1) \Rightarrow du_2/du_1 = \cos(u_1)$- Layer 3: $y = u_2^2 \Rightarrow dy/du_2 = 2u_2$$$\frac{dy}{dx} = 2\sin(3x) \cdot \cos(3x) \cdot 3 = 6\sin(3x)\cos(3x)$$> 💡 This layer-by-layer decomposition is **exactly how backpropagation works** in neural networks (Notebook 35-36).

### Comprehension Check ✅1. Differentiate $f(x) = (2x^3-1)^4$2. Why can't you just use the power rule on $(2x^3-1)^4$ directly?<details><summary>💡 Explanation after a retrieval attempt</summary>1. $f'(x)=4(2x^3-1)^3 \cdot 6x^2 = 24x^2(2x^3-1)^3$2. The power rule applies to $x^n$, not $(\text{stuff})^n$. The chain rule accounts for the 'stuff' inside.</details>

---## Phase 0 — Math Foundation Practice

In [ ]:
def chain_rule_gradient(df_dg, dg_dx):    '''Given the outer derivative and inner derivative, compute the overall gradient.'''    # YOUR CODE HERE    pass# Test: d/dx[sin(x^2)] at x=1 → cos(1)*2 ≈ 1.0806# import math# assert abs(chain_rule_gradient(math.cos(1), 2*1) - 2*math.cos(1)) < 1e-10# print('Phase 0 passed ✅')

---## Phase 1 — Algorithm Construction### Step 1: Numerical Chain RuleGiven two functions $f$ and $g$, compute $\frac{d}{dx}[f(g(x))]$ numerically.

In [ ]:
def composite_derivative(f, g, x, dx=1e-7):    '''Compute d/dx[f(g(x))] numerically.'''    # YOUR CODE HERE    pass# Test: d/dx[sin(x^2)] at x=1# import math# result = composite_derivative(math.sin, lambda x: x**2, 1.0)# expected = 2 * math.cos(1)  # chain rule# assert abs(result - expected) < 1e-5# print('Step 1 passed ✅')

### Step 2: Symbolic Chain Rule EngineBuild a function that applies the chain rule symbolically using SymPy.

In [ ]:
import sympy as spdef apply_chain_rule(outer_func, inner_expr, var=sp.Symbol('x')):    '''Apply chain rule: d/dx[outer(inner(x))]    outer_func: a SymPy function (e.g. sp.sin)    inner_expr: a SymPy expression in var    Returns: the derivative expression    '''    # YOUR CODE HERE    pass# Test:# x = sp.Symbol('x')# result = apply_chain_rule(sp.sin, x**2)# print(result)  # should be 2*x*cos(x**2)

### Step 3: Multi-Layer Chain (Interleaved Practice)Implement a function that chains N derivatives together.

In [ ]:
def multi_chain(derivatives_list):    '''Given [df/dg, dg/dh, dh/dx, ...], multiply them all.    This is the chain rule for arbitrarily deep compositions.    '''    result = 1    for d in derivatives_list:        # YOUR CODE HERE        pass    return result

---## Phase 2 — Experimental Verification### Pathological Case: Vanishing Gradient in Deep ChainsWhen composing many functions with small derivatives, the product of gradients → 0.

In [ ]:
import numpy as np, matplotlib.pyplot as plt# Simulate a chain of N sigmoid-like functions# Each local gradient is at most 0.25 (sigmoid'(0) = 0.25)def sigmoid_grad(x): return np.exp(-x) / (1+np.exp(-x))**2depths = range(1, 51)final_grads = [sigmoid_grad(0)**d for d in depths]plt.figure(figsize=(9,5))plt.semilogy(list(depths), final_grads, 'r-o', ms=3)plt.xlabel('Chain Depth (number of layers)')plt.ylabel('Gradient magnitude (log scale)')plt.title('Vanishing Gradient Problem: Chain Rule with Sigmoid')plt.grid(alpha=0.3); plt.tight_layout(); plt.show()print(f'After 10 layers: gradient ≈ {sigmoid_grad(0)**10:.2e}')print(f'After 50 layers: gradient ≈ {sigmoid_grad(0)**50:.2e}')print('\n⚠️ This is the VANISHING GRADIENT problem — it makes deep neural networks hard to train!')

---## Phase 3 — Olympiad Extension### Challenge: ReLU vs Sigmoid Gradient FlowThe ReLU function $\max(0,x)$ has gradient 1 for $x>0$ and 0 for $x\leq 0$.Show that a chain of ReLU gradients does NOT vanish (but can die completely).Implement both and compare.

In [ ]:
# YOUR CODE: compare gradient flow through 50 sigmoid vs 50 ReLU layers# Plot both on the same graphpass

### Chalkboard Defense1. Prove the chain rule using the limit definition2. Explain why vanishing gradients are a problem for neural network training3. Why does ReLU partially solve this?<details><summary>💡 Sketch after a retrieval attempt</summary>$\frac{d}{dx}f(g(x))=\lim_{h\to0}\frac{f(g(x+h))-f(g(x))}{h}$. Let $k=g(x+h)-g(x)$, then $=\lim\frac{f(g(x)+k)-f(g(x))}{k}\cdot\frac{k}{h}=f'(g(x))\cdot g'(x)$.</details>---📚 **Next:** Math 04 (Local Extrema) — uses derivatives to find peaks and valleys

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.math_03_calculus_chain_rule', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='math_04_calculus_local_extrema.ipynb')